# MobileADAS3D — MobileNetV4 Conv Small baseline

This notebook trains a fresh monocular-3D baseline on the canonical KITTI Chen 3,712/3,769 split. It keeps the existing stride-16 FPN and eight 3D heads, changes only the backbone to pretrained MobileNetV4 Conv Small, saves checkpoints to Google Drive, resumes after disconnects, and produces KITTI BEV/3D AP_R40 artifacts.

Before running: select **Runtime → Change runtime type → GPU** and make sure the repository changes containing this notebook are pushed to GitHub.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, shlex, subprocess, sys

REPO_URL = 'https://github.com/Ali-RT/mobile_adas3d.git'
BRANCH = 'main'
PROJECT_DIR = Path('/content/mobile_adas3d')
CONFIG = 'configs/kitti_mnv4_conv_small_baseline.yaml'
DRIVE_DATASET_ROOT = Path('/content/drive/MyDrive/datasets/kitti')
RUNTIME_DATASET_ROOT = Path('/content/kitti')
SPLIT_DIR = Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
OUTPUT_DIR = Path('/content/drive/MyDrive/mobile_adas3d_outputs/mnv4_conv_small_baseline')
STAGE_DATA_TO_LOCAL = True  # Faster epochs; source data remains in Drive.
FORCE_RESTAGE_DATA = False  # Set True only when you want rsync to repair/re-copy local KITTI.
AUTO_RESUME = True

def run(command, cwd=None):
    print('+', ' '.join(shlex.quote(str(x)) for x in command))
    subprocess.run([str(x) for x in command], cwd=cwd, check=True)


In [ ]:
if not (PROJECT_DIR / '.git').exists():
    run(['git', 'clone', '--branch', BRANCH, REPO_URL, PROJECT_DIR])
else:
    run(['git', 'fetch', 'origin'], cwd=PROJECT_DIR)
    run(['git', 'checkout', BRANCH], cwd=PROJECT_DIR)
    run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=PROJECT_DIR)
os.chdir(PROJECT_DIR)
print('Repository:', PROJECT_DIR)
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], cwd=PROJECT_DIR)
import torch, timm
print('torch:', torch.__version__)
print('timm:', timm.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU is not enabled. Select Runtime > Change runtime type > GPU.')
print('GPU:', torch.cuda.get_device_name(0))


## Stage KITTI onto the Colab runtime

Training directly from mounted Drive can bottleneck the data loader. This stages KITTI to temporary Colab storage for faster epochs while checkpoints and metrics remain in Drive. The staging helper prints source/local counts, shows a per-folder file-count progress bar, uses resumable `rsync --partial --stats`, skips a complete local copy, and writes `/content/kitti/.mobileadas3d_stage_manifest.json`. If Colab disconnects during staging, rerun this cell. Set `STAGE_DATA_TO_LOCAL=False` only if local disk space is insufficient.

In [ ]:
if STAGE_DATA_TO_LOCAL:
    stage_command = [
        sys.executable,
        'scripts/stage_colab_kitti.py',
        '--source', DRIVE_DATASET_ROOT,
        '--destination', RUNTIME_DATASET_ROOT,
    ]
    if FORCE_RESTAGE_DATA:
        stage_command.append('--force')
    run(stage_command, cwd=PROJECT_DIR)
    DATASET_ROOT = RUNTIME_DATASET_ROOT
else:
    DATASET_ROOT = DRIVE_DATASET_ROOT
print('Training dataset root:', DATASET_ROOT)


## Canonical split and full preflight

This fails before training unless all 7,481 images, labels, and calibration files exist; the split is exactly 3,712/3,769 with no overlap; CUDA works; pretrained MobileNetV4 loads; output shapes remain stride 16; and a real KITTI loss is finite.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
run([sys.executable, 'scripts/prepare_kitti_chen_split.py', '--config', CONFIG, '--profile', 'colab_drive', '--output-dir', SPLIT_DIR], cwd=PROJECT_DIR)
COMMON = ['--config', CONFIG, '--profile', 'colab_drive', '--dataset-root', DATASET_ROOT, '--split-dir', SPLIT_DIR, '--output-dir', OUTPUT_DIR]
run([sys.executable, 'scripts/check_kitti_splits.py', *COMMON], cwd=PROJECT_DIR)
run([sys.executable, 'scripts/check_training_ready.py', *COMMON, '--require-cuda', '--report', OUTPUT_DIR / 'training_preflight.json'], cwd=PROJECT_DIR)


## Train or resume

`latest.pt` is atomically replaced after every epoch. With `AUTO_RESUME=True`, rerunning this cell after a Colab disconnect continues the newest run in this dedicated output directory. Epoch snapshots are retained every 10 epochs.

In [ ]:
latest_candidates = list((OUTPUT_DIR / 'runs').glob('*/checkpoints/latest.pt'))
resume_checkpoint = max(latest_candidates, key=lambda p: p.stat().st_mtime) if (AUTO_RESUME and latest_candidates) else None
train_command = [sys.executable, 'scripts/train_mobile_adas3d.py', *COMMON]
if resume_checkpoint is not None:
    train_command += ['--resume', resume_checkpoint]
    TRAIN_RUN_DIR = resume_checkpoint.parent.parent
    print('Resuming:', resume_checkpoint)
else:
    print('Starting a new baseline run')
run(train_command, cwd=PROJECT_DIR)
if resume_checkpoint is None:
    run_dirs = list((OUTPUT_DIR / 'runs').glob('*'))
    TRAIN_RUN_DIR = max(run_dirs, key=lambda p: p.stat().st_mtime)
BEST_CHECKPOINT = TRAIN_RUN_DIR / 'checkpoints' / 'best.pt'
if not BEST_CHECKPOINT.is_file():
    raise FileNotFoundError(f'Best checkpoint missing: {BEST_CHECKPOINT}')
print('Run directory:', TRAIN_RUN_DIR)
print('Best checkpoint:', BEST_CHECKPOINT)


## Generate the reportable KITTI baseline

This evaluates all 3,769 validation images at a low score floor and writes `kitti_r40_metrics.csv`, `kitti_r40_summary.json`, and raw KITTI-format predictions. Only results with `complete_split: true` are reportable.

In [ ]:
EVAL_DIR = TRAIN_RUN_DIR / 'kitti_r40_val'
run([sys.executable, 'scripts/evaluate_kitti_r40.py', '--config', CONFIG, '--profile', 'colab_drive', '--dataset-root', DATASET_ROOT, '--split-dir', SPLIT_DIR, '--checkpoint', BEST_CHECKPOINT, '--split', 'val', '--score-threshold', '0.001', '--topk', '300', '--nms-iou-threshold', '0.5', '--output-dir', EVAL_DIR], cwd=PROJECT_DIR)

import json, pandas as pd
summary = json.loads((EVAL_DIR / 'kitti_r40_summary.json').read_text())
assert summary['complete_split'] and summary['evaluated_images'] == 3769
metrics = pd.DataFrame(summary['metrics'])
display(metrics.pivot_table(index=['metric', 'class_name'], columns='difficulty', values='ap_r40').round(3))
print('Baseline artifacts:', EVAL_DIR)


## Optional TensorBoard

Run the following cell while training or after it finishes.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/mobile_adas3d_outputs/mnv4_conv_small_baseline/runs
